# Лабораторная работа № 5
## Расчёт метрик и подготовка данных для анализа

**Проект:** Bike Sharing Dataset — анализ спроса на прокат велосипедов  
**Выполнили:** Попов Никита и Гордий Владимир  
**Группа:** ИВТ-241

### Исследовательский вопрос из Project Brief

**Как различается количество аренд велосипедов в зависимости от сезона и погодных условий?**

Источник расчётов — готовая аналитическая витрина `hourly_rentals`, сформированная в ЛР № 4. Гранулярность витрины не меняется: одна строка соответствует одному часовому наблюдению (`dteday` + `hr`).

## 1. Метрики и правила расчёта

**Основные метрики (две разные метрики из Project Brief):**

1. `avg_rentals_per_hour` — среднее количество аренд за одно часовое наблюдение, `AVG(rentals_count)`.
2. `median_rentals_per_hour` — медианное количество аренд за одно часовое наблюдение, `MEDIAN(rentals_count)`.

Дополнительные показатели:

- `total_rentals` — сумма количества аренд, `SUM(rentals_count)`. Используется для описания общего объёма и расчёта доли сезона.
- `hourly_observations` — размер группы, `COUNT(*)`.
- `rentals_values_count` — число непустых значений меры, `COUNT(rentals_count)`. Оно показывает фактический размер выборки, использованной функциями `AVG` и `MEDIAN`.

SQL-агрегаты `AVG`, `MEDIAN` и `SUM` не учитывают `NULL` в столбце меры. Размер группы (`COUNT(*)`) и число значений меры (`COUNT(rentals_count)`) поэтому выводятся отдельно. В отдельную числовую выборку включаются только строки с непустым `rentals_count`; это условие отбора будет проверено и зафиксировано. Известный ноль остаётся полноценным числовым значением и не исключается.

**Использование флагов качества:** поля `quality_flag_ids` и `quality_flag_reason` сохраняются в источнике. В этом ноутбуке строки по флагам не отбрасываются: сначала рассчитывается и выводится число помеченных строк, а группировки выполняются по всей витрине. Это сохраняет исходную совокупность наблюдений и соответствует тому, что в сохранённой проверке ЛР № 4 помеченных строк не было.

**Период:** определяется запросом по `MIN(dteday)` и `MAX(dteday)`. Дата `2026-10-18` в S3-префиксе — дата загрузки объекта, а не дата наблюдений.

## 2. Поиск корня проекта и параметры S3

Используется та же схема поиска корня проекта и чтения локального `.env`, что и в `build_mart.ipynb` (ЛР № 4). Секреты подключения не выводятся и не записываются в ноутбук.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Ищет корень проекта по pyproject.toml и .env.example."""
    current = (start or Path.cwd()).resolve()
    candidates = (current, *current.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / ".env.example").is_file():
            return candidate
    raise FileNotFoundError(
        "Не удалось найти корень проекта: среди текущего каталога и его родителей "
        "нет одновременно pyproject.toml и .env.example."
    )


def read_env_file(path: Path) -> dict[str, str]:
    """Читает локальный .env без вывода его содержимого."""
    values: dict[str, str] = {}
    if not path.is_file():
        raise FileNotFoundError(
            f"Локальный .env не найден: {path}. Создайте его из .env.example; "
            "не добавляйте .env в Git."
        )
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def sql_literal(value: str) -> str:
    """Безопасно заключает строковое значение в SQL-литерал DuckDB."""
    return "'" + value.replace("'", "''") + "'"


PROJECT_ROOT = find_project_root()
ENV_PATH = PROJECT_ROOT / ".env"
env = read_env_file(ENV_PATH)

S3_ACCESS_KEY = env.get("S3_ACCESS_KEY", "s3admin")
S3_SECRET_KEY = env.get("S3_SECRET_KEY", "s3admin123")
S3_ENDPOINT = env.get("S3_ENDPOINT", "localhost:8333").strip()
S3_ENDPOINT = (
    S3_ENDPOINT.removeprefix("http://")
    .removeprefix("https://")
    .rstrip("/")
)

DATASET_SLUG = "bike-sharing"
INGESTED_ON = "2026-10-18"
MART_NAME = "hourly_rentals"
MART_PATH = (
    f"s3://mart/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/{MART_NAME}.parquet"
)
EXPECTED_ROW_COUNT = 17_379  # Зафиксированный результат проверки витрины в ЛР № 4.

print("Корень проекта:", PROJECT_ROOT)
print("Источник расчётов:", MART_PATH)
print("Ожидаемое количество строк по результатам ЛР № 4:", EXPECTED_ROW_COUNT)

Корень проекта: C:\itad-lab2
Источник расчётов: s3://mart/bike-sharing/source_ingested_on=2026-10-18/hourly_rentals.parquet
Ожидаемое количество строк по результатам ЛР № 4: 17379


## 3. Подключение DuckDB и чтение витрины

Воспроизводятся настройки `httpfs`, S3 endpoint, ключей, региона и стиля адресации из ЛР № 4. Витрина читается непосредственно из Parquet в S3. `hive_partitioning = false` отключает автоматическое добавление полей из Hive-пути к физической схеме Parquet.

In [2]:
con = duckdb.connect()

try:
    con.execute("LOAD httpfs")
except Exception:
    # Установка выполняется только если расширение ещё не доступно в этой среде.
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")

con.execute(f"SET s3_endpoint={sql_literal(S3_ENDPOINT)}")
con.execute(f"SET s3_access_key_id={sql_literal(S3_ACCESS_KEY)}")
con.execute(f"SET s3_secret_access_key={sql_literal(S3_SECRET_KEY)}")
con.execute("SET s3_region='us-east-1'")
con.execute("SET s3_url_style='path'")
con.execute("SET s3_use_ssl=false")

con.execute(
    f"CREATE OR REPLACE TEMP VIEW mart_data AS "
    f"SELECT * FROM read_parquet({sql_literal(MART_PATH)}, hive_partitioning = false)"
)

print("DuckDB:", duckdb.__version__)
print("Соединение с источником создано; временное представление mart_data доступно.")

DuckDB: 1.1.3
Соединение с источником создано; временное представление mart_data доступно.


## 4. Проверка схемы и первичный профиль источника

До расчётов проверяется фактический состав витрины. Ожидается та же схема из 18 полей, которая была проверена в ЛР № 4. При несовпадении схема не будет молча принята: выполнение остановится с диагностикой.

In [3]:
expected_schema = [
    ("dteday", "DATE"),
    ("hr", "INTEGER"),
    ("season", "INTEGER"),
    ("weathersit", "INTEGER"),
    ("yr", "INTEGER"),
    ("mnth", "INTEGER"),
    ("weekday", "INTEGER"),
    ("workingday", "INTEGER"),
    ("holiday", "INTEGER"),
    ("temp", "DOUBLE"),
    ("atemp", "DOUBLE"),
    ("hum", "DOUBLE"),
    ("windspeed", "DOUBLE"),
    ("rentals_count", "BIGINT"),
    ("casual_rentals_count", "BIGINT"),
    ("registered_rentals_count", "BIGINT"),
    ("quality_flag_ids", "VARCHAR"),
    ("quality_flag_reason", "VARCHAR"),
]

mart_schema = con.execute("DESCRIBE mart_data").df()
actual_schema = [
    (row["column_name"], row["column_type"].upper())
    for _, row in mart_schema.iterrows()
]
schema_matches = actual_schema == expected_schema

schema_check = pd.DataFrame([
    {
        "position": position,
        "expected_field": expected_name,
        "expected_type": expected_type,
        "actual_field": actual_schema[position - 1][0] if position <= len(actual_schema) else None,
        "actual_type": actual_schema[position - 1][1] if position <= len(actual_schema) else None,
        "matches": position <= len(actual_schema)
            and actual_schema[position - 1] == (expected_name, expected_type),
    }
    for position, (expected_name, expected_type) in enumerate(expected_schema, start=1)
])

print("Фактическое число полей:", len(actual_schema))
print("Схема полностью соответствует ЛР № 4:", schema_matches)
display(schema_check)

if not schema_matches:
    raise ValueError(
        "Схема витрины отличается от ожидаемой схемы ЛР № 4. "
        f"Ожидалось: {expected_schema}; фактически: {actual_schema}"
    )

Фактическое число полей: 18
Схема полностью соответствует ЛР № 4: True


,position,expected_field,expected_type,actual_field,actual_type,matches
0,1,dteday,DATE,dteday,DATE,True
1,2,hr,INTEGER,hr,INTEGER,True
2,3,season,INTEGER,season,INTEGER,True
3,4,weathersit,INTEGER,weathersit,INTEGER,True
4,5,yr,INTEGER,yr,INTEGER,True
5,6,mnth,INTEGER,mnth,INTEGER,True
6,7,weekday,INTEGER,weekday,INTEGER,True
7,8,workingday,INTEGER,workingday,INTEGER,True
8,9,holiday,INTEGER,holiday,INTEGER,True
9,10,temp,DOUBLE,temp,DOUBLE,True


In [4]:
source_profile = con.execute(
    """
    SELECT
        COUNT(*) AS hourly_observations,
        COUNT(rentals_count) AS rentals_values_count,
        COUNT(*) - COUNT(rentals_count) AS rentals_null_count,
        MIN(dteday) AS first_observation_date,
        MAX(dteday) AS last_observation_date,
        MIN(rentals_count) AS min_rentals_count,
        MAX(rentals_count) AS max_rentals_count,
        COUNT(*) FILTER (WHERE rentals_count < 0) AS negative_rentals_rows,
        COUNT(*) FILTER (WHERE season IS NULL) AS null_season_rows,
        COUNT(*) FILTER (WHERE weathersit IS NULL) AS null_weather_rows,
        COUNT(*) FILTER (
            WHERE season IS NOT NULL AND season NOT IN (1, 2, 3, 4)
        ) AS invalid_season_rows,
        COUNT(*) FILTER (
            WHERE weathersit IS NOT NULL AND weathersit NOT IN (1, 2, 3, 4)
        ) AS invalid_weather_rows
    FROM mart_data
    """
).df()

display(source_profile)

,hourly_observations,rentals_values_count,rentals_null_count,first_observation_date,last_observation_date,min_rentals_count,max_rentals_count,negative_rentals_rows,null_season_rows,null_weather_rows,invalid_season_rows,invalid_weather_rows
0,17379,17379,0,2011-01-01,2012-12-31,1,977,0,0,0,0,0


In [5]:
quality_profile = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(COALESCE(quality_flag_ids, '')), '') IS NOT NULL
               OR NULLIF(TRIM(COALESCE(quality_flag_reason, '')), '') IS NOT NULL
        ) AS rows_with_quality_flags,
        COUNT(*) FILTER (WHERE quality_flag_ids IS NULL) AS null_quality_flag_ids,
        COUNT(*) FILTER (WHERE quality_flag_reason IS NULL) AS null_quality_flag_reason
    FROM mart_data
    """
).df()

print("Профиль флагов качества. Строки не исключаются из расчётов:")
display(quality_profile)

Профиль флагов качества. Строки не исключаются из расчётов:


,total_rows,rows_with_quality_flags,null_quality_flag_ids,null_quality_flag_reason
0,17379,0,0,0


## 5. Общий итог по всей витрине

Общий итог вычисляется непосредственно по исходным часовым наблюдениям, а не усреднением групповых средних. `AVG` и `MEDIAN` рассчитываются по непустым значениям `rentals_count`; количество этих значений показано отдельно.

In [6]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT
        COUNT(*) AS hourly_observations,
        COUNT(rentals_count) AS rentals_values_count,
        COUNT(*) - COUNT(rentals_count) AS rentals_null_count,
        COALESCE(SUM(rentals_count), 0) AS total_rentals,
        AVG(rentals_count) AS avg_rentals_per_hour,
        MEDIAN(rentals_count) AS median_rentals_per_hour,
        MIN(rentals_count) AS min_rentals_per_hour,
        MAX(rentals_count) AS max_rentals_per_hour,
        MIN(dteday) AS first_observation_date,
        MAX(dteday) AS last_observation_date
    FROM mart_data
    """
)

overall_metrics_result = con.execute(
    """
    SELECT
        hourly_observations,
        rentals_values_count,
        rentals_null_count,
        total_rentals,
        ROUND(avg_rentals_per_hour, 2) AS avg_rentals_per_hour,
        ROUND(median_rentals_per_hour, 2) AS median_rentals_per_hour,
        min_rentals_per_hour,
        max_rentals_per_hour,
        first_observation_date,
        last_observation_date
    FROM overall_metrics
    """
).df()

display(overall_metrics_result)

,hourly_observations,rentals_values_count,rentals_null_count,total_rentals,avg_rentals_per_hour,median_rentals_per_hour,min_rentals_per_hour,max_rentals_per_hour,first_observation_date,last_observation_date
0,17379,17379,0,3292679.0,189.46,142.0,1,977,2011-01-01,2012-12-31


## 6. Основная группировка — метрики по сезонам

Для каждого сезона рассчитываются те же две основные метрики, общий объём аренд и размеры группы. Оконная функция `SUM(...) OVER ()` рассчитывает общий объём аренд по всем сезонным группам. Доля сезона определяется как его сумма, делённая на этот общий объём. При нулевом или отсутствующем знаменателе доля возвращается как `NULL`, а не вызывает деление на ноль.

В таблицу включаются все группы, в том числе возможная группа с `season IS NULL`: скрытая фильтрация категорий не применяется.

In [7]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE metrics_by_season AS
    WITH season_aggregates AS (
        SELECT
            season,
            COUNT(*) AS hourly_observations,
            COUNT(rentals_count) AS rentals_values_count,
            SUM(rentals_count) AS total_rentals,
            AVG(rentals_count) AS avg_rentals_per_hour,
            MEDIAN(rentals_count) AS median_rentals_per_hour
        FROM mart_data
        GROUP BY season
    ),
    season_with_denominator AS (
        SELECT
            *,
            SUM(total_rentals) OVER () AS all_seasons_total_rentals
        FROM season_aggregates
    )
    SELECT
        season,
        hourly_observations,
        rentals_values_count,
        total_rentals,
        avg_rentals_per_hour,
        median_rentals_per_hour,
        all_seasons_total_rentals,
        CASE
            WHEN all_seasons_total_rentals IS NULL
              OR all_seasons_total_rentals = 0
                THEN NULL
            ELSE 100.0 * total_rentals / all_seasons_total_rentals
        END AS season_share_pct
    FROM season_with_denominator
    """
)

metrics_by_season_result = con.execute(
    """
    SELECT
        season,
        hourly_observations,
        rentals_values_count,
        total_rentals,
        ROUND(avg_rentals_per_hour, 2) AS avg_rentals_per_hour,
        ROUND(median_rentals_per_hour, 2) AS median_rentals_per_hour,
        all_seasons_total_rentals,
        ROUND(season_share_pct, 4) AS season_share_pct
    FROM metrics_by_season
    ORDER BY season NULLS LAST
    """
).df()

display(metrics_by_season_result)

,season,hourly_observations,rentals_values_count,total_rentals,avg_rentals_per_hour,median_rentals_per_hour,all_seasons_total_rentals,season_share_pct
0,1,4242,4242,471348.0,111.11,76.0,3292679.0,14.3150
1,2,4409,4409,918589.0,208.34,165.0,3292679.0,27.8979
2,3,4496,4496,1061129.0,236.02,199.0,3292679.0,32.2269
3,4,4232,4232,841613.0,198.87,155.5,3292679.0,25.5601


## 7. Уточняющая группировка — сезон и погодные условия

Вторая группировка уточняет основной вопрос: метрики рассчитываются для каждой наблюдаемой комбинации `season` и `weathersit`. Набор метрик остаётся тем же — сумма, среднее и медиана количества аренд. Размер группы и число непустых значений выводятся отдельно.

Комбинации, которых нет в исходной витрине, искусственно не создаются. Если в исходных данных есть `NULL` в признаках группировки, такие записи остаются в соответствующих группах.

In [8]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE metrics_by_season_weather AS
    SELECT
        season,
        weathersit,
        COUNT(*) AS hourly_observations,
        COUNT(rentals_count) AS rentals_values_count,
        SUM(rentals_count) AS total_rentals,
        AVG(rentals_count) AS avg_rentals_per_hour,
        MEDIAN(rentals_count) AS median_rentals_per_hour
    FROM mart_data
    GROUP BY season, weathersit
    """
)

metrics_by_season_weather_result = con.execute(
    """
    SELECT
        season,
        weathersit,
        hourly_observations,
        rentals_values_count,
        total_rentals,
        ROUND(avg_rentals_per_hour, 2) AS avg_rentals_per_hour,
        ROUND(median_rentals_per_hour, 2) AS median_rentals_per_hour
    FROM metrics_by_season_weather
    ORDER BY season NULLS LAST, weathersit NULLS LAST
    """
).df()

display(metrics_by_season_weather_result)

,season,weathersit,hourly_observations,rentals_values_count,total_rentals,avg_rentals_per_hour,median_rentals_per_hour
0,1,1,2665,2665,325278.0,122.06,86.0
1,1,2,1205,1205,124205.0,103.07,73.0
2,1,3,369,369,21642.0,58.65,35.0
3,1,4,3,3,223.0,74.33,36.0
4,2,1,2859,2859,664463.0,232.41,192.0
5,2,2,1144,1144,207363.0,181.26,143.0
6,2,3,406,406,46763.0,115.18,67.5
7,3,1,3280,3280,801941.0,244.49,210.0
8,3,2,947,947,212932.0,224.85,188.0
9,3,3,269,269,46256.0,171.96,121.0


## 8. Отдельная выборка числовых значений для ЛР № 7

Таблица `hourly_rental_values` содержит исходные значения `rentals_count` на уровне отдельного часового наблюдения и признаки, позволяющие связать значение с датой, сезоном и погодой. Для числового распределения выбираются только записи, где `rentals_count IS NOT NULL`. Значение `0`, если оно встретится, сохраняется. Фильтр по флагам качества или по сезону/погоде не применяется.

Полная выборка остаётся доступной в памяти ноутбука. Для проверки и удобства просмотра ниже выводятся её количество и первые десять строк.

In [9]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE hourly_rental_values AS
    SELECT
        dteday,
        hr,
        season,
        weathersit,
        rentals_count
    FROM mart_data
    WHERE rentals_count IS NOT NULL
    """
)

numeric_sample_profile = con.execute(
    """
    SELECT
        COUNT(*) AS numeric_observations,
        COUNT(DISTINCT rentals_count) AS distinct_rentals_values,
        MIN(rentals_count) AS min_rentals_count,
        MAX(rentals_count) AS max_rentals_count,
        AVG(rentals_count) AS avg_rentals_count,
        MEDIAN(rentals_count) AS median_rentals_count
    FROM hourly_rental_values
    """
).df()

print("Профиль числовой выборки:")
display(numeric_sample_profile)

print("Первые 10 строк числовой выборки (полная таблица hourly_rental_values остаётся в памяти):")
display(
    con.execute(
        """
        SELECT dteday, hr, season, weathersit, rentals_count
        FROM hourly_rental_values
        ORDER BY dteday, hr
        LIMIT 10
        """
    ).df()
)

Профиль числовой выборки:


,numeric_observations,distinct_rentals_values,min_rentals_count,max_rentals_count,avg_rentals_count,median_rentals_count
0,17379,869,1,977,189.463088,142.0


Первые 10 строк числовой выборки (полная таблица hourly_rental_values остаётся в памяти):


,dteday,hr,season,weathersit,rentals_count
0,2011-01-01,0,1,1,16
1,2011-01-01,1,1,1,40
2,2011-01-01,2,1,1,32
3,2011-01-01,3,1,1,13
4,2011-01-01,4,1,1,1
5,2011-01-01,5,1,2,1
6,2011-01-01,6,1,1,2
7,2011-01-01,7,1,1,3
8,2011-01-01,8,1,1,8
9,2011-01-01,9,1,1,14


## 9. Проверки расчётов

Проверки сопоставляют групповые результаты с общим итогом, рассчитанным напрямую по `mart_data`. В частности, проверяются суммы и размеры групп в обеих группировках, сумма долей, количество значений в отдельной числовой выборке и контрольная формула среднего `SUM / COUNT`.

In [10]:
validation_values = con.execute(
    """
    SELECT
        (SELECT hourly_observations FROM overall_metrics) AS overall_rows,
        (SELECT rentals_values_count FROM overall_metrics) AS overall_valid_values,
        (SELECT total_rentals FROM overall_metrics) AS overall_total_rentals,
        (SELECT avg_rentals_per_hour FROM overall_metrics) AS overall_average,
        (SELECT COALESCE(SUM(hourly_observations), 0) FROM metrics_by_season) AS season_group_rows,
        (SELECT COALESCE(SUM(rentals_values_count), 0) FROM metrics_by_season) AS season_valid_values,
        (SELECT COALESCE(SUM(total_rentals), 0) FROM metrics_by_season) AS season_total_rentals,
        (SELECT COALESCE(SUM(hourly_observations), 0) FROM metrics_by_season_weather) AS season_weather_group_rows,
        (SELECT COALESCE(SUM(rentals_values_count), 0) FROM metrics_by_season_weather) AS season_weather_valid_values,
        (SELECT COALESCE(SUM(total_rentals), 0) FROM metrics_by_season_weather) AS season_weather_total_rentals,
        (SELECT SUM(season_share_pct) FROM metrics_by_season) AS season_share_sum_pct,
        (SELECT COUNT(*) FROM hourly_rental_values) AS numeric_sample_rows
    """
).df()

display(validation_values)

,overall_rows,overall_valid_values,overall_total_rentals,overall_average,season_group_rows,season_valid_values,season_total_rentals,season_weather_group_rows,season_weather_valid_values,season_weather_total_rentals,season_share_sum_pct,numeric_sample_rows
0,17379,17379,3292679.0,189.463088,17379.0,17379.0,3292679.0,17379.0,17379.0,3292679.0,100.0,17379


In [11]:
values = validation_values.iloc[0]

overall_rows = int(values["overall_rows"])
overall_valid_values = int(values["overall_valid_values"])
overall_total_rentals = int(values["overall_total_rentals"])
season_share_sum = values["season_share_sum_pct"]

# Проверка долей: при нулевом знаменателе SQL возвращает NULL для каждой доли.
if overall_total_rentals == 0:
    shares_sum_to_100 = pd.isna(season_share_sum)
else:
    shares_sum_to_100 = (
        not pd.isna(season_share_sum)
        and abs(float(season_share_sum) - 100.0) <= 1e-8
    )

# Независимая арифметическая сверка общего среднего с суммой и числом значений.
if overall_valid_values == 0:
    average_matches_sum_over_count = pd.isna(values["overall_average"])
else:
    expected_average = overall_total_rentals / overall_valid_values
    average_matches_sum_over_count = (
        not pd.isna(values["overall_average"])
        and abs(float(values["overall_average"]) - expected_average) <= 1e-9
    )

validation_checks = [
    (
        "Количество строк в витрине соответствует результату ЛР № 4 (17 379)",
        overall_rows == EXPECTED_ROW_COUNT,
        f"фактически {overall_rows}; ожидалось {EXPECTED_ROW_COUNT}",
    ),
    (
        "Сумма размеров групп по сезонам равна общему числу строк",
        int(values["season_group_rows"]) == overall_rows,
        f"по группам {int(values['season_group_rows'])}; всего {overall_rows}",
    ),
    (
        "Сумма размеров групп по сезону и погоде равна общему числу строк",
        int(values["season_weather_group_rows"]) == overall_rows,
        f"по группам {int(values['season_weather_group_rows'])}; всего {overall_rows}",
    ),
    (
        "Сумма аренд по сезонам равна общему количеству аренд",
        int(values["season_total_rentals"]) == overall_total_rentals,
        f"по сезонам {int(values['season_total_rentals'])}; всего {overall_total_rentals}",
    ),
    (
        "Сумма аренд по сезону и погоде равна общему количеству аренд",
        int(values["season_weather_total_rentals"]) == overall_total_rentals,
        f"по детальным группам {int(values['season_weather_total_rentals'])}; всего {overall_total_rentals}",
    ),
    (
        "Сумма долей сезонов равна 100% (или корректно NULL при нулевом знаменателе)",
        bool(shares_sum_to_100),
        f"сумма долей: {season_share_sum}",
    ),
    (
        "Число непустых значений меры в основной группировке сохранено",
        int(values["season_valid_values"]) == overall_valid_values,
        f"по сезонам {int(values['season_valid_values'])}; всего непустых значений {overall_valid_values}",
    ),
    (
        "Число непустых значений меры в уточняющей группировке сохранено",
        int(values["season_weather_valid_values"]) == overall_valid_values,
        f"по детальным группам {int(values['season_weather_valid_values'])}; всего непустых значений {overall_valid_values}",
    ),
    (
        "Число строк числовой выборки равно числу непустых значений rentals_count",
        int(values["numeric_sample_rows"]) == overall_valid_values,
        f"в выборке {int(values['numeric_sample_rows'])}; непустых значений {overall_valid_values}",
    ),
    (
        "В поле season нет пропусков",
        int(source_profile.loc[0, "null_season_rows"]) == 0,
        f"пропусков season: {int(source_profile.loc[0, 'null_season_rows'])}",
    ),
    (
        "В поле weathersit нет пропусков",
        int(source_profile.loc[0, "null_weather_rows"]) == 0,
        f"пропусков weathersit: {int(source_profile.loc[0, 'null_weather_rows'])}",
    ),
    (
        "Коды сезонов входят в допустимый набор {1, 2, 3, 4}",
        int(source_profile.loc[0, "invalid_season_rows"]) == 0,
        f"недопустимых кодов сезона: {int(source_profile.loc[0, 'invalid_season_rows'])}",
    ),
    (
        "Коды погодных условий входят в допустимый набор {1, 2, 3, 4}",
        int(source_profile.loc[0, "invalid_weather_rows"]) == 0,
        f"недопустимых кодов погоды: {int(source_profile.loc[0, 'invalid_weather_rows'])}",
    ),
    (
        "Количество аренд не содержит отрицательных значений",
        int(source_profile.loc[0, "negative_rentals_rows"]) == 0,
        f"строк с отрицательным rentals_count: {int(source_profile.loc[0, 'negative_rentals_rows'])}",
    ),
    (
        "Общее среднее совпадает с SUM(rentals_count) / COUNT(rentals_count)",
        bool(average_matches_sum_over_count),
        "сверка выполнена по исходным значениям",
    ),
]

checks_summary = pd.DataFrame([
    {
        "check": name,
        "result": "PASS" if passed else "FAIL",
        "details": details,
    }
    for name, passed, details in validation_checks
])

display(checks_summary)

failed_checks = checks_summary.loc[checks_summary["result"] != "PASS"]
print(f"Пройдено проверок: {(checks_summary['result'] == 'PASS').sum()} из {len(checks_summary)}")

if not failed_checks.empty:
    raise AssertionError(
        "Не все проверки расчётов пройдены. Изучите строки FAIL в checks_summary "
        "и найдите причину до сдачи лабораторной."
    )

print("Все проверки расчётов пройдены.")

,check,result,details
0,Количество строк в витрине соответствует резул...,PASS,фактически 17379; ожидалось 17379
1,Сумма размеров групп по сезонам равна общему ч...,PASS,по группам 17379; всего 17379
2,Сумма размеров групп по сезону и погоде равна ...,PASS,по группам 17379; всего 17379
3,Сумма аренд по сезонам равна общему количеству...,PASS,по сезонам 3292679; всего 3292679
4,Сумма аренд по сезону и погоде равна общему ко...,PASS,по детальным группам 3292679; всего 3292679
5,Сумма долей сезонов равна 100% (или корректно ...,PASS,сумма долей: 100.0
6,Число непустых значений меры в основной группи...,PASS,по сезонам 17379; всего непустых значений 17379
7,Число непустых значений меры в уточняющей груп...,PASS,по детальным группам 17379; всего непустых зна...
8,Число строк числовой выборки равно числу непус...,PASS,в выборке 17379; непустых значений 17379
9,В поле season нет пропусков,PASS,пропусков season: 0


Пройдено проверок: 15 из 15
Все проверки расчётов пройдены.


## 10. Итоговая проверка доступности результатов

Перед завершением ноутбука убеждаемся, что все три обязательные временные таблицы созданы в текущей сессии DuckDB. Они предназначены для последующего использования в ЛР № 7 и не записываются в отдельные файлы.

In [12]:
required_result_tables = [
    "metrics_by_season",
    "metrics_by_season_weather",
    "hourly_rental_values",
]

# Проверяем таблицы прямым обращением к ним, чтобы не зависеть от особенностей
# отображения временных объектов в системном каталоге DuckDB.
result_table_rows = []
for table_name in required_result_tables:
    row_count = int(con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0])
    result_table_rows.append({
        "temporary_table": table_name,
        "available_in_current_session": True,
        "rows": row_count,
    })

results_availability = pd.DataFrame(result_table_rows)
display(results_availability)

if len(results_availability) != 3 or not results_availability["available_in_current_session"].all():
    raise RuntimeError("Не все обязательные временные таблицы созданы.")

print("Три результата ЛР № 5 доступны в памяти DuckDB.")
print("Для сохранения актуальных выходов перезапустите kernel и выполните ноутбук сверху вниз.")

,temporary_table,available_in_current_session,rows
0,metrics_by_season,True,4
1,metrics_by_season_weather,True,13
2,hourly_rental_values,True,17379


Три результата ЛР № 5 доступны в памяти DuckDB.
Для сохранения актуальных выходов перезапустите kernel и выполните ноутбук сверху вниз.
